In [2]:
##import packages and check setup
import sys
import pandas as pd
import requests

print(f"Interpreter: {sys.executable}")
print(f"Pandas version: {pd.__version__}")
print(f"Requests version: {requests.__version__}")
print("\nSetup complete: Ready to process Census FIPS data.")

Interpreter: c:\Users\wolfd\miniconda3\envs\flu_thesis\python.exe
Pandas version: 3.0.6
Requests version: 2.34.2

Setup complete: Ready to process Census FIPS data.


I will pull census data to find est. population for all counties.

In [5]:
## census dataset import

# 1) load the local census dataset
csv_filename = "co-est2024-alldata.csv"

# census text files use 'latin-1' encoding due to special characters in place names
df_raw = pd.read_csv(csv_filename, encoding='latin-1')
print(f"Total raw rows loaded: {len(df_raw):,}")

# 2) remove state level totals
df_counties = df_raw[df_raw['COUNTY'] != 0].copy()

# 3) create the 5-digit zero-padded FIPS code
df_counties['fips'] = (
    df_counties['STATE'].astype(str).str.zfill(2) + 
    df_counties['COUNTY'].astype(str).str.zfill(3)
)

# 4) retain core columns
df_pop = df_counties[['fips', 'STNAME', 'CTYNAME', 'POPESTIMATE2024']].rename(
    columns={
        'STNAME': 'state_name',
        'CTYNAME': 'county_name',
        'POPESTIMATE2024': 'pop_2024'
    }
)

# 5) quick summary
print(f"Valid counties processed: {len(df_pop):,}")
print(f"Total 2024 population: {df_pop['pop_2024'].sum():,}")

df_pop.head(10)

Total raw rows loaded: 3,195
Valid counties processed: 3,144
Total 2024 population: 340,110,988


,fips,state_name,county_name,pop_2024
1,01001,Alabama,Autauga County,61464
2,01003,Alabama,Baldwin County,261608
3,01005,Alabama,Barbour County,24358
4,01007,Alabama,Bibb County,22258
5,01009,Alabama,Blount County,60163
6,01011,Alabama,Bullock County,9901
7,01013,Alabama,Butler County,18256
8,01015,Alabama,Calhoun County,116427
9,01017,Alabama,Chambers County,33813
10,01019,Alabama,Cherokee County,26138


In [6]:
## export baseline pop table

# 1) define output path
output_path = "census_2024_county_population_baseline.csv"

# 2) write to csv, remove dataframe index
df_pop.to_csv(output_path, index=False)
print(f"saved {len(df_pop):,} cleaned county records to {output_path}")

saved 3,144 cleaned county records to census_2024_county_population_baseline.csv


In [7]:
## data validation
# 1) check FIPS characters
invalid_fips_len = df_pop[df_pop['fips'].str.len() != 5]
print(f"records with invalid fips length: {len(invalid_fips_len)}")

# 2) check FIPS dupes
duplicate_fips = df_pop[df_pop['fips'].duplicated()]
print(f"duplicate fips codes found: {len(duplicate_fips)}")

# 3) pop values - non-null and positive values
null_pops = df_pop['pop_2024'].isnull().sum()
zero_or_neg_pops = len(df_pop[df_pop['pop_2024'] <= 0])
print(f"null population counts: {null_pops}")
print(f"counties with zero or negative population: {zero_or_neg_pops}")

records with invalid fips length: 0
duplicate fips codes found: 0
null population counts: 0
counties with zero or negative population: 0


Next, I will pull the geographic points (centroid) for each county using Gazetteer Files (https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.2024.html#form-dropdown-264479560)

In [9]:
## census gazetteer centroids

# 1) load file
gaz_filename = "2024_Gaz_counties_national.txt"

df_gaz_raw = pd.read_csv(gaz_filename, sep='\t', encoding='latin-1', dtype={'GEOID': str})

# 2) clean headers
df_gaz_raw.columns = df_gaz_raw.columns.str.strip()

# 3) extract fips, latitude, and longitude
df_coords = df_gaz_raw[['GEOID', 'INTPTLAT', 'INTPTLONG']].copy()
df_coords.columns = ['fips', 'latitude', 'longitude']

# 4) standardize fips and clean coordinates
df_coords['fips'] = df_coords['fips'].str.strip().str.zfill(5)
df_coords['latitude'] = pd.to_numeric(df_coords['latitude'].astype(str).str.strip())
df_coords['longitude'] = pd.to_numeric(df_coords['longitude'].astype(str).str.strip())

# 5) remove PR to standardize with other census file
df_coords = df_coords[~df_coords['fips'].str.startswith('72')].copy()

# 6) display coordinate summary
print(f"gazetteer records loaded: {len(df_coords):,}")
df_coords.head(5)

gazetteer records loaded: 3,144


,fips,latitude,longitude
0,01001,32.532237,-86.646440
1,01003,30.659218,-87.746067
2,01005,31.870253,-85.405104
3,01007,33.015893,-87.127148
4,01009,33.977358,-86.566440


In [11]:
## merge population with coordinates
# 1) merge based on FIPS
df_baseline = pd.merge(df_pop, df_coords, on='fips', how='inner')

# 2) verify
print(f"merged baseline records: {len(df_baseline):,}")

# 3) check nulls
null_counts = df_baseline.isnull().sum().to_dict()
print(f"null values per column: {null_counts}")

# 4) preview
df_baseline.head(5)

merged baseline records: 3,144
null values per column: {'fips': 0, 'state_name': 0, 'county_name': 0, 'pop_2024': 0, 'latitude': 0, 'longitude': 0}


,fips,state_name,county_name,pop_2024,latitude,longitude
0,01001,Alabama,Autauga County,61464,32.532237,-86.646440
1,01003,Alabama,Baldwin County,261608,30.659218,-87.746067
2,01005,Alabama,Barbour County,24358,31.870253,-85.405104
3,01007,Alabama,Bibb County,22258,33.015893,-87.127148
4,01009,Alabama,Blount County,60163,33.977358,-86.566440


df_baseline now contains all the unique county FIPS, pop. as of 2024, and centroid points of each county. I will now export this table.

In [12]:
## export

# 1) define output path
baseline_output_path = "county_population_and_coordinates_2024.csv"

# 2) write to csv, remove index
df_baseline.to_csv(baseline_output_path, index=False)
print(f"saved {len(df_baseline):,} records to {baseline_output_path}")

# 3) display summ. stats
print("\nsummary overview:")
print(f"total national population: {df_baseline['pop_2024'].sum():,}")
print(f"latitude range: {df_baseline['latitude'].min():.4f} to {df_baseline['latitude'].max():.4f}")
print(f"longitude range: {df_baseline['longitude'].min():.4f} to {df_baseline['longitude'].max():.4f}")

saved 3,144 records to county_population_and_coordinates_2024.csv

summary overview:
total national population: 340,110,988
latitude range: 19.5978 to 69.4493
longitude range: -164.1889 to 179.6212
